### 1. Εγκατάσταση βιβλιοθηκών

In [ ]:
!pip install transformers datasets librosa torchaudio accelerate matplotlib captum

### 2. Imports και ρυθμίσεις

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification, AutoModel

from datasets import load_dataset, Audio, DatasetDict

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

ORIGINAL_MODEL = "facebook/data2vec-audio-base-960h"                          #προεκπαιδευμένο μοντέλο αναφοράς
STAGES_REPO    = "VasilisAsim/data2vec-finetuned-TESS-v2"                     #M0, M1, M4 (branches)
FINAL_REPO     = "VasilisAsim/data2vec-finetuned-TESS_full"                   #Baseline model
EPOCH_REPO_ID  = "VasilisAsim/data2vec-finetuned-TESS-per-checkpoint-ft-v2"   #M3 ανά epoch (branches)
ENCODER_ATTR   = "data2vec_audio"

### 3. Ορισμός checkpoints

In [ ]:
CHECKPOINTS = [
    {"name": "original",   "source": ORIGINAL_MODEL, "revision": None},
    {"name": "M0_frozen",  "source": STAGES_REPO, "revision": "M0-frozen-head"},
    {"name": "M1_partial", "source": STAGES_REPO, "revision": "M1-partial"},
] + [
    {"name": f"M3_epoch{e}", "source": EPOCH_REPO_ID, "revision": f"M3-epoch-{e}"}
    for e in range(1, 9)
] + [
    {"name": "M4_best",        "source": STAGES_REPO, "revision": "M2-full"},
    {"name": "Baseline_final", "source": FINAL_REPO,  "revision": None},
]

STAGE_NAMES = ["original", "M0_frozen", "M1_partial", "M4_best", "Baseline_final"]
EPOCH_NAMES = [f"M3_epoch{e}" for e in range(1, 9)]
STAGE_COLORS = {"original": "tab:gray", "M0_frozen": "tab:blue", "M1_partial": "tab:orange",
                "M4_best": "tab:green", "Baseline_final": "tab:red"}

### 4. Φόρτωση TESS και split ανεξάρτητο ομιλητή

In [ ]:
def load_tess_speaker_independent():
    dataset = load_dataset("AbstractTTS/TESS", split="train")
    dataset = dataset.cast_column("audio", Audio(decode=False))

    def get_speaker(example):
        filename = example["audio"]["path"]
        if "OAF" in filename:
            return "OAF"
        elif "YAF" in filename:
            return "YAF"
        return "unknown"

    train_dataset = dataset.filter(lambda x: get_speaker(x) == "OAF")
    test_dataset = dataset.filter(lambda x: get_speaker(x) == "YAF")

    train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
    test_dataset = test_dataset.cast_column("audio", Audio(sampling_rate=16_000))

    return DatasetDict({"train": train_dataset, "test": test_dataset})


dataset = load_tess_speaker_independent()
train_dataset = dataset["train"]        # OAF (εκπαίδευση του probe)
test_dataset = dataset["test"]          # YAF (αξιολόγηση του probe)

print(f"Train (OAF): {len(train_dataset)} δείγματα")
print(f"Test  (YAF): {len(test_dataset)} δείγματα")

### 5. Συναρτήσεις φόρτωσης μοντέλου και εξαγωγής embeddings

In [ ]:
def load_for_extraction(source, revision=None):
    """Φορτώνει feature extractor + μοντέλο. Επιστρέφει (model, feature_extractor, encoder_attr).
    - Το προεκπαιδευμένο μοντέλο φορτώνεται ως AutoModel (χωρίς κεφαλή ταξινόμησης).
    - Τα fine-tuned φορτώνονται ως AutoModelForAudioClassification και χρησιμοποιείται ο backbone τους."""
    feature_extractor = AutoFeatureExtractor.from_pretrained(source, revision=revision)
    if source == ORIGINAL_MODEL:
        model, encoder_attr = AutoModel.from_pretrained(source), None
    else:
        model, encoder_attr = AutoModelForAudioClassification.from_pretrained(source, revision=revision), ENCODER_ATTR
    return model.to(device).eval(), feature_extractor, encoder_attr


@torch.no_grad()
def extract_embeddings(model, feature_extractor, dataset, encoder_attr=None):
    """Mean pooling στον χρόνο των hidden states κάθε layer, ένα δείγμα τη φορά (χωρίς padding).
    Επιστρέφει (λίστα [layer] -> np.array (δείγματα x hidden_dim), np.array με τις ετικέτες).
    Layer 0 = έξοδος embeddings."""
    backbone = getattr(model, encoder_attr) if encoder_attr else model
    n_layers = model.config.num_hidden_layers + 1
    buffers = [[] for _ in range(n_layers)]
    labels = []

    for sample in dataset:
        audio = sample["audio"]
        inputs = feature_extractor(audio["array"], sampling_rate=audio["sampling_rate"],
                                   return_tensors="pt").to(device)
        out = backbone(inputs["input_values"], output_hidden_states=True)
        for layer, h in enumerate(out.hidden_states):
            buffers[layer].append(h.squeeze(0).mean(dim=0).cpu().numpy())
        labels.append(sample["emotion"])

    return [np.stack(b) for b in buffers], np.array(labels)

### 6. Layer-wise probing

In [ ]:
def evaluate_layer(X_train, y_train, X_test, y_test, num_classes):
    """
    Εκπαιδεύει γραμμικό probe (Logistic Regression) στο σύνολο δοκιμής και αξιολογεί στο σύνολο εκπαίδευσης
    για ένα layer. UAR = balanced accuracy.
    """
    probe = LogisticRegression(max_iter=2000, multi_class="multinomial")
    probe.fit(X_train, y_train)

    y_pred = probe.predict(X_test)
    y_proba = probe.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    bal_acc = balanced_accuracy_score(y_test, y_pred)  # = UAR
    uar = bal_acc  # ταυτόσημο, το κρατάμε ως ξεχωριστό όνομα για σαφήνεια στη γραφή

    # Macro AUC (One-vs-Rest) - χρειάζεται one-hot ground truth
    y_test_bin = label_binarize(y_test, classes=range(num_classes))
    try:
        auc = roc_auc_score(y_test_bin, y_proba, average="macro", multi_class="ovr")
    except ValueError:
        auc = np.nan  # αν λείπει κάποια κλάση από το test split

    return {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "UAR": uar,
        "balanced_accuracy": bal_acc,
        "macro_auc": auc
    }

### 7. Εξαγωγή embeddings και probing για όλα τα checkpoints

In [ ]:
le = LabelEncoder().fit(train_dataset["emotion"])
num_classes = len(le.classes_)

probing_rows = []
for ckpt in CHECKPOINTS:
    print(f"Probing: {ckpt['name']} ({ckpt['source']}@{ckpt['revision']}) ...")
    model, fe, attr = load_for_extraction(ckpt["source"], ckpt["revision"])
    train_emb, train_lab = extract_embeddings(model, fe, train_dataset, attr)
    test_emb, test_lab = extract_embeddings(model, fe, test_dataset, attr)
    del model
    torch.cuda.empty_cache()

    y_train, y_test = le.transform(train_lab), le.transform(test_lab)
    for layer in range(len(train_emb)):
        metrics = evaluate_layer(train_emb[layer], y_train, test_emb[layer], y_test, num_classes)
        probing_rows.append({"checkpoint": ckpt["name"], "layer": layer, **metrics})

    # ενδιάμεση αποθήκευση μετά από κάθε checkpoint (προστασία από διακοπή runtime)
    pd.DataFrame(probing_rows).to_csv("probing_results_data2vec_tess.csv", index=False)

probing_results = pd.DataFrame(probing_rows)

### 8. Πίνακας αποτελεσμάτων (layers × checkpoints)

In [ ]:
METRICS = ["accuracy", "macro_f1", "UAR", "macro_auc"]
ORDER = [c["name"] for c in CHECKPOINTS]

probing_tables = {
    m: probing_results.pivot(index="layer", columns="checkpoint", values=m)[ORDER]
    for m in METRICS
}
probing_tables["accuracy"].round(3)      # άλλαξε το κλειδί για τις υπόλοιπες μετρικές

### 9. Οπτικοποίηση 1: Probing ανά στάδιο (original / M0 / M1 / M4 / Baseline)

In [ ]:
def plot_probing(names, colors, title, filename):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
    for ax, metric in zip(axes.flat, METRICS):
        for name, color in zip(names, colors):
            sub = probing_results[probing_results["checkpoint"] == name]
            ax.plot(sub["layer"], sub[metric], marker="o", label=name, color=color)
        ax.set_title(metric)
        ax.set_xlabel("Layer index (0 = embeddings, 12 = τελευταίο transformer layer)")
        ax.set_ylabel(metric)
        ax.set_ylim(0, 1.02)
        ax.grid(alpha=0.3)
    handles, lbls = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, lbls, loc="upper center", ncol=len(names), bbox_to_anchor=(0.5, 1.03), fontsize=8)
    fig.suptitle(title, y=1.08)
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()


plot_probing(STAGE_NAMES, [STAGE_COLORS[n] for n in STAGE_NAMES],
             "Layer-wise probing ανά στάδιο (data2vec, TESS)",
             "probing_stages_data2vec_tess.png")

### 10. Οπτικοποίηση 2: Probing ανά epoch (στάδιο M3)

In [ ]:
plot_probing(EPOCH_NAMES, plt.cm.viridis(np.linspace(0, 1, len(EPOCH_NAMES))),
             "Layer-wise probing ανά epoch (data2vec, TESS, M3)",
             "probing_epochs_data2vec_tess.png")